In [3]:
import sqlite3
import pandas as pd

# Tạo database file banking.db
conn = sqlite3.connect("banking.db")

# Load 4 CSV vào database
users        = pd.read_csv("data/users.csv")
app_events   = pd.read_csv("data/app_events.csv")
transactions = pd.read_csv("data/transactions.csv")
churn_labels = pd.read_csv("data/churn_labels.csv")

# Ghi vào SQLite
users.to_sql("users",        conn, if_exists="replace", index=False)
app_events.to_sql("app_events",   conn, if_exists="replace", index=False)
transactions.to_sql("transactions", conn, if_exists="replace", index=False)
churn_labels.to_sql("churn_labels", conn, if_exists="replace", index=False)

# Verify
for table in ["users", "app_events", "transactions", "churn_labels"]:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn).iloc[0,0]
    print(f"{table}: {count} rows")

print("\n✅ banking.db created successfully!")

users: 5000 rows
app_events: 27927 rows
transactions: 10523 rows
churn_labels: 5000 rows

✅ banking.db created successfully!


In [5]:
query = """
SELECT
    (SELECT COUNT(DISTINCT user_id) FROM users) AS total_users,

    ROUND(100.0 *
        (SELECT COUNT(DISTINCT user_id) FROM app_events
         WHERE event_name = 'account_activated' AND status = 'success')
        / (SELECT COUNT(DISTINCT user_id) FROM users), 1
    ) AS activation_rate_pct,

    ROUND(100.0 *
        (SELECT SUM(is_active_d30) FROM churn_labels)
        / (SELECT COUNT(DISTINCT user_id) FROM users), 1
    ) AS d30_retention_pct,

    ROUND(100.0 *
        (SELECT SUM(churn_flag) FROM churn_labels)
        / (SELECT COUNT(DISTINCT user_id) FROM users), 1
    ) AS churn_rate_pct
"""

pd.read_sql(query, conn)

,total_users,activation_rate_pct,d30_retention_pct,churn_rate_pct
0,5000,34.5,24.6,85.1


In [6]:
query = """
SELECT
    event_name,
    COUNT(DISTINCT user_id)  AS users_reached,
    ROUND(
        100.0 * COUNT(DISTINCT user_id)
        / (SELECT COUNT(DISTINCT user_id) FROM app_events
           WHERE event_name = 'app_install'), 1
    ) AS pct_of_installs
FROM app_events
WHERE event_name IN (
    'app_install', 'register_start', 'register_complete',
    'ekyc_start', 'ekyc_complete', 'account_activated'
)
AND status = 'success'
GROUP BY event_name
ORDER BY
    CASE event_name
        WHEN 'app_install'       THEN 1
        WHEN 'register_start'    THEN 2
        WHEN 'register_complete' THEN 3
        WHEN 'ekyc_start'        THEN 4
        WHEN 'ekyc_complete'     THEN 5
        WHEN 'account_activated' THEN 6
    END
"""

pd.read_sql(query, conn)

,event_name,users_reached,pct_of_installs
0,app_install,5000,100.0
1,register_start,4617,92.3
2,register_complete,3886,77.7
3,ekyc_start,3467,69.3
4,ekyc_complete,2300,46.0
5,account_activated,1725,34.5


In [7]:
query = """
SELECT
    u.acquisition_channel,
    COUNT(DISTINCT u.user_id)  AS total_users,
    COUNT(DISTINCT CASE WHEN e.event_name = 'account_activated'
                         AND e.status = 'success'
                        THEN e.user_id END)  AS activated_users,
    ROUND(
        100.0 * COUNT(DISTINCT CASE WHEN e.event_name = 'account_activated'
                                     AND e.status = 'success'
                                    THEN e.user_id END)
        / COUNT(DISTINCT u.user_id), 1
    )  AS activation_rate_pct
FROM users u
LEFT JOIN app_events e ON u.user_id = e.user_id
GROUP BY u.acquisition_channel
ORDER BY activation_rate_pct DESC
"""

pd.read_sql(query, conn)

,acquisition_channel,total_users,activated_users,activation_rate_pct
0,Referral,974,434,44.6
1,Organic,1197,480,40.1
2,Branch,512,199,38.9
3,Partnership,747,256,34.3
4,Paid Ads,1570,356,22.7


In [8]:
query = """
WITH first_txn AS (
    -- Tìm ngày transaction đầu tiên của mỗi user
    SELECT
        user_id,
        MIN(txn_date) AS first_txn_date
    FROM transactions
    WHERE status = 'success'
    GROUP BY user_id
),
activation AS (
    -- Tìm ngày activated của mỗi user
    SELECT
        user_id,
        MIN(event_time) AS activation_date
    FROM app_events
    WHERE event_name = 'account_activated' AND status = 'success'
    GROUP BY user_id
),
user_speed AS (
    -- Tính số ngày từ activation đến first transaction
    SELECT
        a.user_id,
        CAST(
            (JULIANDAY(f.first_txn_date) - JULIANDAY(a.activation_date))
        AS INTEGER) AS days_to_first_txn
    FROM activation a
    LEFT JOIN first_txn f ON a.user_id = f.user_id
)
SELECT
    CASE
        WHEN days_to_first_txn IS NULL THEN 'No transaction'
        WHEN days_to_first_txn <= 7    THEN 'Within 7 days'
        ELSE 'After 7 days'
    END                                          AS txn_timing,
    COUNT(*)                                     AS users,
    ROUND(100.0 * SUM(cl.is_active_d30)
          / COUNT(*), 1)                         AS d30_retention_pct
FROM user_speed us
JOIN churn_labels cl ON us.user_id = cl.user_id
GROUP BY txn_timing
ORDER BY d30_retention_pct DESC
"""

pd.read_sql(query, conn)

,txn_timing,users,d30_retention_pct
0,Within 7 days,781,78.1
1,After 7 days,684,73.8
2,No transaction,260,43.8


In [9]:
query = """
WITH activated_users AS (
    SELECT DISTINCT user_id
    FROM app_events
    WHERE event_name = 'account_activated' AND status = 'success'
)
SELECT
    event_name                            AS feature,
    COUNT(DISTINCT user_id)               AS users_adopted,
    ROUND(
        100.0 * COUNT(DISTINCT user_id)
        / (SELECT COUNT(*) FROM activated_users), 1
    )                                     AS adoption_rate_pct
FROM app_events
WHERE event_name IN (
    'transfer_initiated', 'qr_payment', 'bill_payment',
    'savings_opened', 'card_payment', 'loan_viewed'
)
AND status = 'success'
AND user_id IN (SELECT user_id FROM activated_users)
GROUP BY event_name
ORDER BY adoption_rate_pct DESC
"""

pd.read_sql(query, conn)

,feature,users_adopted,adoption_rate_pct
0,savings_opened,635,36.8
1,loan_viewed,631,36.6
2,card_payment,605,35.1
3,qr_payment,601,34.8
4,bill_payment,599,34.7
5,transfer_initiated,586,34.0


In [10]:
query = """
WITH activated_users AS (
    SELECT DISTINCT user_id
    FROM app_events
    WHERE event_name = 'account_activated' AND status = 'success'
),
user_features AS (
    SELECT
        user_id,
        COUNT(DISTINCT event_name)                                      AS n_features,
        MAX(CASE WHEN event_name = 'bill_payment'   THEN 1 ELSE 0 END) AS uses_bill,
        MAX(CASE WHEN event_name = 'savings_opened' THEN 1 ELSE 0 END) AS uses_savings
    FROM app_events
    WHERE event_name IN (
        'transfer_initiated', 'qr_payment', 'bill_payment',
        'savings_opened', 'card_payment', 'loan_viewed'
    )
    AND status = 'success'
    AND user_id IN (SELECT user_id FROM activated_users)
    GROUP BY user_id
)
SELECT
    CASE
        WHEN uf.uses_bill = 1 OR uf.uses_savings = 1 THEN 'Bill/Savings user'
        WHEN uf.n_features >= 2                       THEN 'Multi-feature (no bill/savings)'
        WHEN uf.n_features = 1                        THEN 'Single feature'
        ELSE 'No feature used'
    END                                               AS user_segment,
    COUNT(*)                                          AS users,
    ROUND(100.0 * SUM(cl.is_active_d30) / COUNT(*), 1) AS d30_retention_pct
FROM activated_users au
LEFT JOIN user_features uf ON au.user_id = uf.user_id
LEFT JOIN churn_labels  cl ON au.user_id = cl.user_id
GROUP BY user_segment
ORDER BY d30_retention_pct DESC
"""

pd.read_sql(query, conn)

,user_segment,users,d30_retention_pct
0,Bill/Savings user,998,79.6
1,Multi-feature (no bill/savings),276,75.0
2,Single feature,276,58.3
3,No feature used,175,38.3


In [11]:
query = """
WITH user_failed_txn AS (
    SELECT
        user_id,
        COUNT(*) AS failed_txns
    FROM transactions
    WHERE status = 'failed'
    GROUP BY user_id
)
SELECT
    CASE
        WHEN ft.failed_txns IS NULL THEN '0 failed'
        WHEN ft.failed_txns <= 2   THEN '1-2 failed'
        ELSE '3+ failed'
    END                                               AS fail_bucket,
    COUNT(*)                                          AS users,
    ROUND(100.0 * SUM(cl.churn_flag) / COUNT(*), 1)  AS churn_rate_pct
FROM churn_labels cl
LEFT JOIN user_failed_txn ft ON cl.user_id = ft.user_id
GROUP BY fail_bucket
ORDER BY churn_rate_pct DESC
"""

pd.read_sql(query, conn)

,fail_bucket,users,churn_rate_pct
0,0 failed,4362,89.8
1,3+ failed,45,73.3
2,1-2 failed,593,51.4


In [12]:
query = """
SELECT
    STRFTIME('%Y-%m', registration_date)        AS cohort_month,
    COUNT(DISTINCT u.user_id)                   AS cohort_size,
    ROUND(100.0 * SUM(cl.is_active_d30)
          / COUNT(DISTINCT u.user_id), 1)       AS d30_retention_pct,
    ROUND(100.0 * SUM(cl.is_active_d60)
          / COUNT(DISTINCT u.user_id), 1)       AS d60_retention_pct
FROM users u
LEFT JOIN churn_labels cl ON u.user_id = cl.user_id
GROUP BY cohort_month
ORDER BY cohort_month
"""

pd.read_sql(query, conn)

,cohort_month,cohort_size,d30_retention_pct,d60_retention_pct
0,2024-01,1269,25.0,15.9
1,2024-02,1287,24.0,14.3
2,2024-03,1362,24.5,14.6
3,2024-04,1082,24.9,15.0
